# Error Analysis

## Imports & Load data

In [7]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score

df_train = pd.read_csv('../data/work/train.csv')
df_test  = pd.read_csv('../data/work/test.csv')
X_train = df_train.drop(columns=['total'])
y_train = df_train['total'].values
X_test  = df_test.drop(columns=['total'])
y_test  = df_test['total'].values

## Build evaluation Dataframe

In [8]:
cat_cols = ["address", "district", "type"]
num_cols = ["bedrooms", "garage"]
area_col = ["area"]  

for c in cat_cols:
    if c in X_test.columns:
        X_test[c] = X_test[c].astype("string").str.strip().str.lower()
        
y_train = pd.to_numeric(y_train, errors="coerce")

p99_area = np.nanpercentile(X_train["area"].to_numpy(), 99)

random_forest = RandomForestRegressor(random_state=42, n_jobs=-1)

area_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clip99", FunctionTransformer(
        lambda X: np.clip(X, None, p99_area),
        feature_names_out="one-to-one"
    )),
])

num_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
])

ct_tree = ColumnTransformer(
    transformers=[
        ("num_other", num_tree, num_cols),
        ("area",      area_tree, area_col),  
        ("cat",       cat_pipe,  cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

pipe_forest_tunned = Pipeline(steps=[
    ("preprocess", ct_tree),       
    ("model", random_forest)
])


param_dist = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__bootstrap": [True],
}

random_search_forest = RandomizedSearchCV(
    estimator=pipe_forest_tunned,                   
    param_distributions=param_dist,                
    n_iter=20,
    scoring="neg_mean_absolute_error",             
    cv=5,
    random_state=42,
    n_jobs=-1,
    error_score=np.nan                              
)

random_search_forest.fit(X_train, y_train)

best_mae_forest   = -random_search_forest.best_score_
best_params_forest = random_search_forest.best_params_
best_forest_pipe   = random_search_forest.best_estimator_

y_pred = best_forest_pipe.predict(X_test)

eval_df = pd.concat([y_train, y_pred], axis=1).join(X_test, how="left")

eval_df["error"] = eval_df["y_true"] - eval_df["y_pred"]
eval_df["abs_error"] = eval_df["error"].abs()

eval_df.head(3)

TypeError: cannot concatenate object of type '<class 'numpy.ndarray'>'; only Series and DataFrame objs are valid